# Clustering in Practice: Who Are Our Viewers?
### AI/ML for Business · Unsupervised learning lab · MovieLens audience segmentation

**The business question.** A streaming service has 610 active users and their rating history. Marketing wants to stop treating everyone the same way. *Are there distinct audience segments, and how should each one be served differently?*

There is **no label column** here. Nobody has told us which segment each user belongs to. That is what makes this **unsupervised** learning: the algorithm finds structure, and we decide whether that structure is useful.

**By the end of this notebook you should be able to:**
1. Turn raw transactions (ratings) into one feature row per customer
2. Explain why **feature scaling** changes which clusters you get
3. Run **K-means** and choose *K* using the **elbow** and **silhouette** methods, and explain why they can disagree
4. Check whether your clusters are **stable** or just artefacts of random starting points
5. Compare K-means with **hierarchical clustering**
6. Turn clusters into **named personas** and **business actions**

**Data:** MovieLens `ml-latest-small` (GroupLens Research, University of Minnesota): about 100k ratings, 610 users, 9.7k movies.

In [ ]:
# --- Setup -------------------------------------------------------------
import os, zipfile, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

SEED = 42
# A colour-blind-checked categorical palette. We also give each cluster its own
# marker shape, so colour is never the only way to tell clusters apart.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
MARKERS = ["o", "s", "^", "D", "P", "X", "v", "*"]
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "xtick.color": "#898781",
    "ytick.color": "#898781", "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.titleweight": "bold", "axes.titlesize": 12, "font.size": 10,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})
pd.set_option("display.precision", 2)

In [ ]:
# --- Load MovieLens (downloads ~1 MB the first time) -------------------------------
DATA_DIR = "ml-latest-small"
URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
if not os.path.exists(f"{DATA_DIR}/ratings.csv"):
    urllib.request.urlretrieve(URL, "ml-latest-small.zip")
    zipfile.ZipFile("ml-latest-small.zip").extractall(".")

ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv")
movies  = pd.read_csv(f"{DATA_DIR}/movies.csv")
print(f"{len(ratings):,} ratings | {ratings.userId.nunique()} users | {movies.movieId.nunique():,} movies")
display(ratings.head(3)); display(movies.head(3))

## 1. Understand the raw material

Each row in `ratings` is a **transaction**: one user rated one movie at one moment. To segment *customers*, we need one row **per customer**. Before we build that, here is how unevenly activity is spread across users.

In [ ]:
per_user = ratings.groupby("userId").size()
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(per_user, bins=np.logspace(np.log10(per_user.min()), np.log10(per_user.max()), 40), color=PALETTE[0], edgecolor="white", linewidth=1)
ax.set_xscale("log"); ax.set_xlabel("Ratings per user (log scale)"); ax.set_ylabel("Users")
ax.set_title("A few heavy raters, a long tail of light ones")
plt.show()
print(f"Median user: {per_user.median():.0f} ratings | Top user: {per_user.max():,} ratings")

**Discussion:** if we used *raw* rating counts as a feature, what would one user with 2,000+ ratings do to a distance-based algorithm? We log-transform volume below for exactly this reason.

## 2. Feature engineering: one row per user

This is where most of the business judgement lives. We describe each user in two ways:

| Feature group | Features | Business meaning |
|---|---|---|
| **Taste** | Share of each user's ratings falling in each genre (shares sum to 1) | *What* they watch |
| **Behaviour** | `log_ratings` (activity), `avg_rating` (generosity), `rating_std` (how discriminating), `avg_movie_year` (classic vs. new) | *How* they engage |

We drop `IMAX` (a format, not a genre) and `(no genres listed)`.

In [ ]:
# Release year sits in the title, e.g. "Toy Story (1995)"
movies["year"] = movies.title.str.extract(r"\((\d{4})\)\s*$")[0].astype(float)

df = ratings.merge(movies[["movieId", "genres", "year"]], on="movieId")

# --- Taste: share of genre tags per user ---
g = df[["userId", "genres"]].assign(genre=df.genres.str.split("|")).explode("genre", ignore_index=True)
g = g[~g.genre.isin(["IMAX", "(no genres listed)"])]
taste = pd.crosstab(g.userId, g.genre, normalize="index")   # each row sums to 1
GENRES = list(taste.columns)

# --- Behaviour ---
behav = df.groupby("userId").agg(
    log_ratings=("rating", lambda s: np.log10(len(s))),
    avg_rating=("rating", "mean"),
    rating_std=("rating", "std"),
    avg_movie_year=("year", "mean"),
)
BEHAV = list(behav.columns)

users = taste.join(behav).fillna(0)
FEATURES = GENRES + BEHAV
print(f"Feature matrix: {users.shape[0]} users × {users.shape[1]} features")
users.head()

## 3. Build intuition: how K-means works

Before the real data, here is the algorithm on a toy 2-D example. K-means repeats two steps:
1. **Assign** each point to its nearest centroid
2. **Update** each centroid to the mean of the points assigned to it

It stops when assignments no longer change. The goal it minimises is **inertia**: the total squared distance from each point to its own centroid.

> Open the companion interactive explorable to step through this yourself.

In [ ]:
Xt, _ = make_blobs(n_samples=300, centers=4, cluster_std=1.1, random_state=7)
rng = np.random.default_rng(6)
cent = Xt[rng.choice(len(Xt), 4, replace=False)]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), sharex=True, sharey=True)
for i, ax in enumerate(axes):
    lab = np.argmin(((Xt[:, None, :] - cent[None]) ** 2).sum(-1), axis=1)
    inertia = ((Xt - cent[lab]) ** 2).sum()
    for k in range(4):
        ax.scatter(*Xt[lab == k].T, s=14, color=PALETTE[k], marker=MARKERS[k], alpha=.75, linewidths=0)
    ax.scatter(*cent.T, s=160, c="#0b0b0b", marker="X", edgecolors="white", linewidths=1.5)
    ax.set_title(f"Iteration {i}  ·  inertia {inertia:,.0f}", fontsize=10)
    new = np.array([Xt[lab == k].mean(0) for k in range(4)])
    for a, b in zip(cent, new):
        ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="->", color="#52514e", lw=1.2))
    cent = new
plt.suptitle("K-means: assign → update → repeat (arrows show where each centroid moves next)", y=1.04)
plt.show()

## 4. Why scaling matters, shown on our own data

K-means uses **Euclidean distance**, so a feature with a large numeric range dominates. Genre shares run from 0 to 1. `avg_movie_year` varies by *decades*. What happens if we cluster without scaling?

In [ ]:
X_raw = users[FEATURES].values
var_share = pd.Series(X_raw.var(0) / X_raw.var(0).sum(), index=FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 3))
top = var_share.head(8)[::-1]
ax.barh(top.index, top.values * 100, color=PALETTE[0], height=0.6)
ax.set_xlabel("% of total variance (unscaled)"); ax.set_title("Unscaled: one feature owns the distance")
ax.grid(axis="y", visible=False)
plt.show()

km_raw = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit(X_raw)
print("Unscaled K=5: average of each feature by cluster")
display(users.assign(cluster=km_raw.labels_).groupby("cluster")[["avg_movie_year", "avg_rating", "Drama", "Comedy", "Action"]].mean())

**What to notice:** without scaling, the clusters split mainly on release year, essentially *"decade buckets"*, and taste barely matters. That isn't a finding about customers. It's a side effect of the units.

**Fix:** standardise every feature to mean 0 and standard deviation 1, so each one gets an equal vote.

> **A trade-off worth discussing:** standardising also *boosts* rare genres like Film-Noir, since a small share for one user becomes a large z-score. Is that what the business wants? There's no universally right answer, and that judgement belongs to you, not the algorithm.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(users[FEATURES])
print("After scaling, every feature has mean ≈ 0 and std ≈ 1:", X.mean().round(3), X.std().round(3))

## 5. Choosing K: elbow vs. silhouette

K-means needs us to choose **K**. There are two common guides:

- **Elbow (inertia):** inertia *always* falls as K grows. Look for the point where extra clusters stop buying much improvement.
- **Silhouette score (−1 to 1):** for each point, compares *how close it is to its own cluster* with *how close it is to the nearest other cluster*. Higher means better-separated clusters.

In [ ]:
Ks = range(2, 11)
inertias, sils = [], []
for k in Ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    inertias.append(km.inertia_); sils.append(silhouette_score(X, km.labels_))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6))
a1.plot(Ks, inertias, marker="o", color=PALETTE[0], lw=2, ms=7)
a1.set_title("Elbow: inertia keeps falling. Where does it flatten?"); a1.set_xlabel("K"); a1.set_ylabel("Inertia")
a2.plot(Ks, sils, marker="o", color=PALETTE[0], lw=2, ms=7)
best = list(Ks)[int(np.argmax(sils))]
a2.annotate(f"highest: K={best}", (best, max(sils)), textcoords="offset points", xytext=(8, 6), color="#52514e")
a2.set_title("Silhouette: higher = better separated"); a2.set_xlabel("K"); a2.set_ylabel("Mean silhouette")
plt.tight_layout(); plt.show()
pd.DataFrame({"K": list(Ks), "inertia": inertias, "silhouette": sils}).round(3).set_index("K").T

**Expect the methods to disagree, and expect low silhouette values.** Real customer data rarely contains neat, well-separated blobs. Tastes blend into each other. The highest silhouette often lands on a small K (2 or 3) that is too coarse to act on.

**This is the key lesson:** *K is a business decision informed by statistics, not a statistical answer.* Ask:
- How many segments can marketing realistically run different playbooks for?
- Is every segment big enough to matter?
- Can we give every segment a name that makes sense?

👉 **Set `K_FINAL` below.** We default to 5. Try 4 and 6 and compare the personas.

In [ ]:
K_FINAL = 5   # ← instructor/student choice: try 4, 5, 6

km = KMeans(n_clusters=K_FINAL, n_init=10, random_state=SEED).fit(X)
users["cluster"] = km.labels_
sizes = users.cluster.value_counts().sort_index()
print(f"K={K_FINAL} | silhouette = {silhouette_score(X, km.labels_):.3f}")
print("Users per cluster:", sizes.to_dict())

## 6. Stability: do we get the same clusters if we run it again?

K-means starts from **random centroids**. With a single run (`n_init=1`), different random seeds can produce different segments. We measure agreement between runs with the **Adjusted Rand Index** (ARI): 1 means identical groupings, and around 0 means no better than chance. Cluster *numbering* doesn't matter here, only who ends up grouped with whom.

In [ ]:
def stability(n_init, runs=10):
    labs = [KMeans(n_clusters=K_FINAL, n_init=n_init, random_state=s).fit_predict(X) for s in range(runs)]
    return np.array([adjusted_rand_score(labs[i], labs[j]) for i in range(runs) for j in range(i + 1, runs)])

s1, s10 = stability(1), stability(10)
print(f"n_init=1  → ARI between runs: mean {s1.mean():.2f}, min {s1.min():.2f}")
print(f"n_init=10 → ARI between runs: mean {s10.mean():.2f}, min {s10.min():.2f}")

**Takeaway:** a single run can mislead you. `n_init=10` runs K-means 10 times and keeps the lowest-inertia result, which is far more stable. If ARI stays low even with `n_init=10`, the data may not have strong cluster structure at that K, which is itself a finding worth reporting.

## 7. See the clusters: PCA projection

We have 22 dimensions and can only plot 2. **PCA** finds the two directions that capture the most variation, so we can project users onto a 2-D map. Clusters that overlap on this map may still be separate in the full 22-D space, because the map is a shadow, not the whole picture.

In [ ]:
pca = PCA(n_components=2, random_state=SEED).fit(X)
P = pca.transform(X); Cp = pca.transform(km.cluster_centers_)

fig, ax = plt.subplots(figsize=(8.5, 6))
for k in range(K_FINAL):
    m = users.cluster.values == k
    ax.scatter(*P[m].T, s=22, color=PALETTE[k], marker=MARKERS[k], alpha=.7, linewidths=0, label=f"Cluster {k} (n={m.sum()})")
ax.scatter(*Cp.T, s=220, c="#0b0b0b", marker="X", edgecolors="white", linewidths=2, label="Centroid", zorder=5)

ev = pca.explained_variance_ratio_ * 100
ax.set_xlabel(f"PC1 ({ev[0]:.0f}% of variance)"); ax.set_ylabel(f"PC2 ({ev[1]:.0f}% of variance)")
ax.set_title("Users on a 2-D map, coloured and shaped by cluster")
ax.legend(loc="center left", bbox_to_anchor=(1.01, .5), frameon=False)
plt.show()
print(f"These 2 axes capture only {ev.sum():.0f}% of the variation. Keep that in mind when judging overlap.\n")
# What drives each axis? (loadings = weight of each original feature in the axis)
load = pd.DataFrame(pca.components_.T, index=FEATURES, columns=["PC1", "PC2"])
for pc in ["PC1", "PC2"]:
    srt = load[pc].sort_values()
    print(f"{pc}:  low end ← " + ", ".join(srt.index[:3]) + "   |   high end → " + ", ".join(srt.index[::-1][:3]))

## 8. From clusters to personas: profiling

A cluster number means nothing to a CMO. We need to describe each cluster in business terms.

**Genre lift** = (cluster's average genre share) ÷ (everyone's average genre share).
- Lift **> 1** means the cluster over-indexes on that genre (blue)
- Lift **< 1** means it under-indexes (red)
- Lift = 1 means it matches the average user (grey)

In [ ]:
overall = users[GENRES].mean()
lift = users.groupby("cluster")[GENRES].mean() / overall
log_lift = np.log2(lift.clip(lower=1e-3))

from matplotlib.colors import LinearSegmentedColormap
div = LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])
lim = min(2.0, np.abs(log_lift.values).max())
fig, ax = plt.subplots(figsize=(13, 0.55 * K_FINAL + 1.6))
sns.heatmap(log_lift, cmap=div, center=0, vmin=-lim, vmax=lim, annot=lift.round(1), fmt="", annot_kws={"size": 8},
            linewidths=2, linecolor="white", cbar_kws={"label": "log₂ lift (0 = average)"}, ax=ax)
ax.set_title("Genre lift by cluster (numbers = lift; 2.0 means twice the average share)")
ax.set_xlabel(""); ax.set_ylabel("Cluster"); ax.grid(False); plt.show()

In [ ]:
# Behaviour, in original (human-readable) units
prof = users.groupby("cluster").agg(
    users=("avg_rating", "size"),
    median_ratings=("log_ratings", lambda s: round(10 ** s.median())),
    avg_rating=("avg_rating", "mean"),
    rating_std=("rating_std", "mean"),
    avg_movie_year=("avg_movie_year", "mean"),
)
prof["share_of_users"] = (prof.users / prof.users.sum() * 100).round(0).astype(int).astype(str) + "%"
prof.loc["ALL"] = [len(users), round(10 ** users.log_ratings.median()), users.avg_rating.mean(),
                   users.rating_std.mean(), users.avg_movie_year.mean(), "100%"]
prof.round(2)

In [ ]:
# A first-draft persona name for each cluster, generated by rules.
# Students should IMPROVE these: naming is the human part of the job.
med_n, med_yr, med_r = users.log_ratings.median(), users.avg_movie_year.median(), users.avg_rating.median()

def draft_persona(k):
    c = users[users.cluster == k]
    top = lift.loc[k].sort_values(ascending=False)
    tg = [x for x in top.index[:2] if top[x] > 1.1]
    activity = "Heavy" if c.log_ratings.median() > med_n + 0.15 else ("Light" if c.log_ratings.median() < med_n - 0.15 else "")
    era = "Classic-leaning" if c.avg_movie_year.median() < med_yr - 3 else ("New-release" if c.avg_movie_year.median() > med_yr + 3 else "")
    mood = "generous raters" if c.avg_rating.mean() > med_r + 0.2 else ("tough critics" if c.avg_rating.mean() < med_r - 0.2 else "")
    taste = " & ".join(tg) + " fans" if tg else "Broad-taste viewers"
    return " ".join(x for x in [activity, era, taste] if x) + (f", {mood}" if mood else "")

for k in range(K_FINAL):
    top3 = lift.loc[k].sort_values(ascending=False).head(3)
    low2 = lift.loc[k].sort_values().head(2)
    print(f"Cluster {k}  ({prof.loc[k, 'users']} users, {prof.loc[k, 'share_of_users']})")
    print(f"  Draft persona : {draft_persona(k)}")
    print(f"  Over-indexes  : " + ", ".join(f"{g} ×{v:.1f}" for g, v in top3.items()))
    print(f"  Under-indexes : " + ", ".join(f"{g} ×{v:.1f}" for g, v in low2.items()))
    print(f"  Behaviour     : median {prof.loc[k, 'median_ratings']} ratings · avg rating {prof.loc[k, 'avg_rating']:.2f} · avg film year {prof.loc[k, 'avg_movie_year']:.0f}\n")

## 9. A second opinion: hierarchical clustering

**Agglomerative (hierarchical) clustering** works bottom-up. It starts with every user as their own cluster and repeatedly merges the two closest clusters. **Ward linkage** merges whichever pair adds the least within-cluster variance, which is the same objective K-means uses.

The **dendrogram** shows the full merge history. Cutting it horizontally gives any number of clusters, so you don't have to choose K before you start. Tall vertical lines mark merges between very different groups, which are natural places to cut.

In [ ]:
Z = linkage(X, method="ward")
fig, ax = plt.subplots(figsize=(12, 4))
dendrogram(Z, truncate_mode="lastp", p=30, color_threshold=Z[-(K_FINAL - 1), 2], above_threshold_color="#898781",
           leaf_font_size=8, ax=ax)
ax.axhline(Z[-(K_FINAL - 1), 2] - 1e-6, color="#0b0b0b", ls="--", lw=1)
ax.text(ax.get_xlim()[1], Z[-(K_FINAL - 1), 2], f"  cut → {K_FINAL} clusters", va="bottom", ha="right", color="#52514e")
ax.set_title("Ward dendrogram (last 30 merges; leaf labels in brackets = number of users)")
ax.set_ylabel("Merge distance"); ax.grid(False); plt.show()

users["hier"] = fcluster(Z, K_FINAL, criterion="maxclust") - 1
print(f"Agreement with K-means (ARI): {adjusted_rand_score(users.cluster, users.hier):.2f}")
pd.crosstab(users.cluster, users.hier, rownames=["K-means"], colnames=["Hierarchical"])

**Reading the crosstab:** each row is a K-means cluster, and each column is a hierarchical cluster. A strong diagonal pattern (after allowing for different numbering) means two different algorithms found similar segments, which makes them more trustworthy. Where they disagree, the segment boundaries are fuzzy.

| | K-means | Hierarchical (Ward) |
|---|---|---|
| Choose K | Before fitting | After, by cutting the tree |
| Scales to millions of users | ✅ Yes | ❌ Memory grows with n² |
| Deterministic | No (random starts) | Yes |
| Cluster shape bias | Round, similar-sized | Depends on linkage |

## 10. From personas to action

A segmentation only has value if it changes a decision. For each cluster, fill in the table, then debate it in your group.

| Cluster | Persona name (yours) | Size | What would you **do** differently? | KPI to watch |
|---|---|---|---|---|
| 0 | | | e.g. homepage row, acquisition budget, push timing, pricing tier | |
| 1 | | | | |
| … | | | | |

**Prompts:**
- **Content acquisition:** which segment would a new licensing deal for classic films serve? Is that segment big enough to justify it?
- **Recommendations:** should a *light, broad-taste* user get the same algorithm as a *heavy niche* fan?
- **Retention:** which segment would you guess is most at risk of churning? What data would you need to check that guess?

In [ ]:
# Export assignments, e.g. to hand to the CRM team
out = users[["cluster"]].assign(persona=users.cluster.map({k: draft_persona(k) for k in range(K_FINAL)}))
out.to_csv("user_segments.csv")
out.persona.value_counts()

## 11. Caveats: what clustering *cannot* tell you

1. **Clusters always exist.** K-means will return K groups even from pure noise. Finding clusters doesn't prove they're real. Stability, a second algorithm, and business sense are how you check.
2. **The features define the segments.** Different features (e.g. viewing time instead of ratings) produce different segments. Feature design is a *business* decision.
3. **Ratings ≠ viewing.** Users who rate a lot may differ from typical viewers. These are segments of *raters*.
4. **Segments drift.** Re-run periodically. Today's "New-release fans" will rate different films next year.
5. **K-means assumes round, similar-sized clusters.** Stretched or uneven groups get split or merged incorrectly (try the explorable's "uneven" and "elongated" datasets).

## 12. Exercises

1. **Change K.** Re-run with `K_FINAL = 4` and `6`. Which segments survive, and which split or merge? Which K would you present to the CMO, and why?
2. **Taste only.** Drop the behaviour features (`FEATURES = GENRES`). How do the personas change? Which version is more useful for *marketing* compared with *content strategy*?
3. **No scaling.** Re-run sections 7–8 on unscaled data. Write two sentences explaining the result to a non-technical manager.
4. **Stretch:** weight the feature groups, e.g. multiply the behaviour columns by 2 after scaling. How should a business decide on weights?